# Milestone 1: data, fixed split, zero-shot baseline


In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
# Clone YOUR GitHub repo (change the URL if your repo name differs)
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .

Cloning into 'trial-eligibility-transformer'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 25 (delta 0), reused 25 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 20.07 KiB | 4.01 MiB/s, done.
/kaggle/working/trial-eligibility-transformer
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which i

In [3]:
# Unit tests (no GPU needed): split rule, leakage, parsing, metrics, truncation
!python -m pytest -q tests

.........                                                                [100%]
9 passed in 1.51s


## 1. Data preparation + split manifest

In [4]:
!python scripts/prepare_data.py

README.md: 100%|███████████████████████████████| 893/893 [00:00<00:00, 4.29MB/s]
data/train-00000-of-00001.parquet: 100%|██████| 176k/176k [00:00<00:00, 343kB/s]
Generating train split: 100%|█████| 1015/1015 [00:00<00:00, 17323.72 examples/s]
raw rows: 1015  columns: ['annotation_id', 'patient_id', 'note', 'trial_id', 'trial_title', 'criterion_type', 'criterion_text', 'gpt4_explanation', 'explanation_correctness', 'gpt4_sentences', 'expert_sentences', 'gpt4_eligibility', 'expert_eligibility', 'training']

leakage checks: {"patient_overlap": 0, "duplicate_note_across_splits": 0, "annotations_assigned": 1015, "annotations_total": 1015}
dropped rows: {"empty_criterion_text": 1}

split  patients  annots  incl  excl empty_ev
train        37     602   215   387      348
val           8     206    66   140      116
test          8     206    75   131      145

label counts per split:
  train: {'included': 88, 'not included': 13, 'excluded': 8, 'not excluded': 291, 'not enough information': 17

## 2. Zero-shot baseline on VALIDATION (test split stays untouched)

In [5]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --split val --tag baseline_qwen25_1p5b

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
tokenizer_config.json: 7.30kB [00:00, 18.6MB/s]
vocab.json: 2.78MB [00:00, 49.0MB/s]
merges.txt: 1.67MB [00:00, 104MB/s]
tokenizer.json: 7.03MB [00:00, 166MB/s]
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 3.20MB/s]
I0000 00:00:1793856575.606088     228 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:09<00:00, 341MB/s]
baseline_qwen25_1p5b:val: 100%|█████████████████| 26/26 [00:46<00:00,  1.80s/it]

n=206  acc=0.189  macroF1=0.143  invalid=0  | evidence microF1=0.185  macroF1(all)=0.379  macroF1(nonempty gold)=0.256
inclusion acc=0.500  exclusion acc=0.043
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'm

In [6]:
!python scripts/evaluate.py --model HuggingFaceTB/SmolLM2-1.7B-Instruct --split val --tag baseline_smollm2_1p7b

tokenizer_config.json: 3.76kB [00:00, 10.1MB/s]
vocab.json: 801kB [00:00, 19.6MB/s]
merges.txt: 466kB [00:00, 65.3MB/s]
tokenizer.json: 2.10MB [00:00, 113MB/s]
config.json: 100%|█████████████████████████████| 908/908 [00:00<00:00, 4.71MB/s]
I0000 00:00:1793856666.448801     289 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100%|████████████████████| 3.42G/3.42G [00:10<00:00, 319MB/s]
baseline_smollm2_1p7b:val: 100%|████████████████| 26/26 [00:34<00:00,  1.33s/it]

n=206  acc=0.199  macroF1=0.150  invalid=0  | evidence microF1=0.094  macroF1(all)=0.089  macroF1(nonempty gold)=0.137
inclusion acc=0.591  exclusion acc=0.014
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 500, 'mean_prompt_tokens': 387.3349514563107}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/results/baseline_smollm2_1p7b


## 3. Comparison table

In [7]:
import json
for tag in ['baseline_qwen25_1p5b', 'baseline_smollm2_1p7b']:
    m = json.load(open(f'results/{tag}/metrics.json'))
    ri = json.load(open(f'results/{tag}/run_info.json'))
    o = m['overall']
    print(f"{tag:28s} acc={o['classification']['accuracy']:.3f} macroF1={o['classification']['macro_f1']:.3f} "
          f"invalid={o['classification']['n_invalid']:3d} evMicroF1={o['evidence']['micro']['f1']:.3f} "
          f"params={ri['model']['n_params']/1e9:.2f}B peakMem={ri['peak_gpu_mem_mb']:.0f}MB time={ri['wall_time_s']:.0f}s")

baseline_qwen25_1p5b         acc=0.189 macroF1=0.143 invalid=  0 evMicroF1=0.185 params=1.54B peakMem=3278MB time=47s
baseline_smollm2_1p7b        acc=0.199 macroF1=0.150 invalid=  0 evMicroF1=0.094 params=1.71B peakMem=4173MB time=35s


In [8]:
# A few raw outputs, to see how the base model behaves
!head -n 3 results/baseline_qwen25_1p5b/predictions.jsonl

{"annotation_id": 0, "patient_id": "sigir-20141", "trial_id": "NCT01397994", "criterion_type": "inclusion", "criterion_text": "Patients of chronic stable angina with abnormal Exercise Myocardial Perfusion Spect Scan with reversible and partially reversible ischemic changes.", "gold_label": "not enough information", "pred_label": "not applicable", "gold_evidence": [0, 1, 2], "pred_evidence": [], "parse_ok": true, "hallucinated_ids": [], "raw_output": "{\"label\": \"not applicable\", \"evidence\": []}", "prompt_tokens": 390, "dropped_sentence_ids": [], "gold_evidence_lost": []}
{"annotation_id": 1, "patient_id": "sigir-20141", "trial_id": "NCT01397994", "criterion_type": "inclusion", "criterion_text": "Male and female", "gold_label": "included", "pred_label": "not applicable", "gold_evidence": [0], "pred_evidence": [], "parse_ok": true, "hallucinated_ids": [], "raw_output": "{\"label\": \"not applicable\", \"evidence\": []}", "prompt_tokens": 369, "dropped_sentence_ids": [], "gold_eviden

## 4. Download results
Download `m1_results.zip` from the Output panel (right side), then unzip it into your local repo and commit, or just send the two `metrics.json` + the manifest to Claude.

In [9]:
!zip -qr /kaggle/working/m1_results.zip data/splits results
!ls -la /kaggle/working

total 48
drwxr-xr-x  4 root root  4096 Nov  5 05:32 .
drwxr-xr-x  5 root root  4096 Nov  5 05:28 ..
-rw-r--r--  1 root root 31709 Nov  5 05:32 m1_results.zip
drwxr-xr-x 11 root root  4096 Nov  5 05:29 trial-eligibility-transformer
drwxr-xr-x  2 root root  4096 Nov  5 05:28 .virtual_documents
